# Agentic Workspace MCP: Example Usage

This notebook demonstrates how to utilize the **Sandboxed Agentic Workspace MCP** server with the **OpenAI Agents SDK** and **OpenRouter**.

### Tech Stack:
- **MCP Server**: Sandboxed environment (Docker).
- **Orchestration**: OpenAI Agents SDK (Python).
- **Model Backend**: OpenRouter (giving you access to Gemini, Claude, etc.).

## 1. Setup Environment
We need to load our API keys from `.env` and configure the event loop for the notebook if necessary.

In [1]:
import os
import asyncio
from pathlib import Path
from dotenv import load_dotenv

# load_dotenv() 
# If using a specific path:
load_dotenv(Path("..") / ".env")

print(f"OpenRouter API Key found: {bool(os.environ.get('OPENROUTER_API_KEY'))}")

OpenRouter API Key found: True


## 2. Initialize MCP Server Connection
We connect to the existing Docker-based MCP server via standard I/O (stdio).

**Security Hardening**: The container is launched with dropped capabilities, restricted memory/CPU, and no privilege escalation.

In [2]:
from agents.mcp import MCPServerStdio

# Configure the server link
# We assume the workspace root is the parent directory's 'tmp'
root_dir = Path(os.getcwd()).parent
abs_tmp_dir = str((root_dir / "tmp").resolve())

mcp_server = MCPServerStdio(
    name="Sandboxed Workspace",
    params={ 
        "command": "docker", 
        "args": [ 
            "run", "-i", "--rm", 
            "--user", "1000:1000", 
            "--security-opt", "no-new-privileges",
            "--cap-drop", "ALL",
            "--init",
            "--memory", "512m",
            "--cpus", "0.5",
            "-v", f"{abs_tmp_dir}:/workspace", 
            "agent-workspace-mcp" 
        ] 
    }, 
    client_session_timeout_seconds=60.0 
)

## 3. Define the Agent
We'll use a specialized model from OpenRouter (e.g., Gemini 2.0 Flash).

In [3]:
from agents import Agent
from agents.extensions.models.litellm_model import LitellmModel

model_name = os.environ.get("DEFAULT_MODEL", "openrouter/google/gemini-2.0-flash-001")

agent = Agent(
    name="WorkspaceAnalyst",
    instructions=(
        "You are a workspace operations analyst. Your goal is to explore the filesystem, "
        "audit artifacts, and perform maintenance tasks using your bash and file tools."
    ),
    model=LitellmModel(model=model_name),
    mcp_servers=[mcp_server]
)

## 4. Run the Agent (Streamed)
The following function runs the agent and prints tool calls, results, and text responses in real-time.

In [4]:
from agents import Runner, RunConfig

async def run_analysis(mission: str):
    print(f"🚀 Starting Mission: {mission}\n")
    
    async with mcp_server:
        # The Runner.run_streamed returns a result object that exposes a stream_events() generator.
        # We use await here as Jupyter inherently supports top-level async.
        stream = Runner.run_streamed(agent, mission, max_turns=15, run_config=RunConfig())
        async for event in stream.stream_events():
            if event.type == "agent_event":
                data = event.data
                if data.type == "tool_call":
                    print(f"\n🛠️  [TOOL CALL] {data.name}({data.arguments})")
                elif data.type == "tool_result":
                    # Truncate large results for readability
                    output = str(data.output)
                    if len(output) > 200: output = output[:200] + "..."
                    print(f"✅ [RESULT] {output}")
                elif data.type == "text_delta":
                    print(data.text_delta, end="", flush=True)
                elif data.type == "handoff":
                    print(f"➡️  [HANDOFF] to {data.agent_name}")
            elif event.type == "control_event":
                pass

mission = "List the contents of the /workspace directory. Pick one .py file, read its metadata (size/mod time), and report its findings. If nothing is there, just a create a nice little python script, execute it and tell about it."
# Use top-level await (supported natively in Jupyter kernels)
await run_analysis(mission)

🚀 Starting Mission: List the contents of the /workspace directory. Pick one .py file, read its metadata (size/mod time), and report its findings. If nothing is there, just a create a nice little python script, execute it and tell about it.

